# 4. Final Model Evaluation

## 4.1 Objective

In this notebook, we evaluate our best Word-Embedding and RNN models on the held-out test dataset.

Both models were selected based on their validation accuracy. The test set was not used during training or hyperparameter optimization.

The goal is to compare how well both architectures generalize to previously unseen examples.

### Selected Models

| Model | Dropout | Weight Decay | Seed | Best Validation Accuracy |
|---|---|---|---|---|
| Word Embeddings | 0.0 | 0.001 | 42 | 67.80% |
| RNN (GRU) | 0.5 | 0.001 | 42 | 66.10% |

The final evaluation uses the saved checkpoints from these experiments. No further hyperparameter tuning will be performed using the test results.

## 4.2 Load Best Model Checkpoints

The best Word-Embedding and RNN (GRU) models are restored from the checkpoints saved during training in `outputs/checkpoints/`. No model is retrained.

The checkpoint file names follow the experiment IDs created by `run_experiment()` in `src/training.py`. Each checkpoint contains the model weights (`model_state_dict`), the vocabulary built from the training data (`vocab`) and the hyperparameters of the experiment.

The RNN checkpoint additionally stores `embedding_dim` and `hidden_dim`. The older Word-Embedding checkpoint does not contain these keys, so the embedding dimension is derived from the shape of the saved embedding weights.

In [1]:
import sys
from pathlib import Path

import torch

# Add the project directory to the Python path
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import RNNModel, WordEmbeddingModel

# Use the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_dir = project_root / "outputs" / "checkpoints"

print("Device:", device)

Device: cuda


In [2]:
def load_model(checkpoint_name, model_type):
    """Rebuild a model from a saved checkpoint and prepare it for evaluation."""
    checkpoint = torch.load(
        checkpoint_dir / checkpoint_name,
        map_location=device,
        weights_only=True
    )

    state_dict = checkpoint["model_state_dict"]
    vocab = checkpoint["vocab"]

    # Architecture parameters: stored in newer checkpoints,
    # otherwise derived from the saved weights
    vocab_size, embedding_dim = state_dict["embedding.weight"].shape
    embedding_dim = checkpoint.get("embedding_dim", embedding_dim)

    if model_type == "rnn":
        hidden_dim = checkpoint.get("hidden_dim", state_dict["rnn.weight_hh_l0"].shape[1])

        model = RNNModel(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            dropout=checkpoint["dropout"]
        )
    elif model_type == "word_embeddings":
        model = WordEmbeddingModel(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            dropout=checkpoint["dropout"]
        )
    else:
        raise ValueError(f"Unknown model type: {model_type!r}")

    # strict=True ensures that all weights match the architecture exactly
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    return model, vocab, checkpoint

In [3]:
# Best Word-Embedding model: BS 256, LR 0.001, Dropout 0.0, WD 0.001, Seed 42
we_model, we_vocab, we_checkpoint = load_model(
    "word_embeddings_bs256_lr0.001_do0_wd0.001_ep60_p5_s42.pt",
    model_type="word_embeddings"
)

# Best RNN (GRU) model: BS 256, LR 0.001, Dropout 0.5, WD 0.001, Seed 42
rnn_model, rnn_vocab, rnn_checkpoint = load_model(
    "rnn_bs256_lr0.001_do0.5_wd0.001_ep60_p5_s42.pt",
    model_type="rnn"
)

# Both models must use the same vocabulary for encoding the test data
assert we_vocab == rnn_vocab, "The checkpoints use different vocabularies"
vocab = we_vocab

for name, model, checkpoint in [
    ("Word Embeddings", we_model, we_checkpoint),
    ("RNN (GRU)", rnn_model, rnn_checkpoint),
]:
    print(
        f"{name:16s} | Best epoch: {checkpoint['epoch']:2d} | "
        f"Validation accuracy: {checkpoint['valid_accuracy']:.2%} | "
        f"Parameters: {sum(p.numel() for p in model.parameters()):,} | "
        f"Device: {next(model.parameters()).device} | "
        f"Eval mode: {not model.training}"
    )

print(f"\nVocabulary size: {len(vocab)}")
print("Both models were loaded successfully and are ready for evaluation.")

Word Embeddings  | Best epoch: 23 | Validation accuracy: 67.80% | Parameters: 800,629 | Device: cuda:0 | Eval mode: True
RNN (GRU)        | Best epoch: 17 | Validation accuracy: 66.10% | Parameters: 825,589 | Device: cuda:0 | Eval mode: True

Vocabulary size: 15818
Both models were loaded successfully and are ready for evaluation.


## 4.3 Test Data Preparation

The held-out test split is loaded and prepared with the same preprocessing functions from `src/preprocessing.py` that were used during training.

The test examples are encoded with the vocabulary loaded from the checkpoints in section 4.2. No new vocabulary is built from the test data. Words that only occur in the test set are mapped to the `<UNK>` token, exactly as unknown words were handled during validation.

The test data is used exclusively for the final evaluation and not for training or model selection. The DataLoader does not shuffle the data, so the order of the test examples is preserved.

In [4]:
from datasets import load_dataset
from torch.utils.data import DataLoader

from src.preprocessing import collate_batch, encode_example

# Load the held-out test split
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

# Encode the test examples with the vocabulary from the checkpoints
test_encoded = [encode_example(example, vocab) for example in test]

test_loader = DataLoader(
    test_encoded,
    batch_size=256,
    shuffle=False,
    collate_fn=collate_batch
)

print("Test examples:", len(test_encoded))
print("Test batches:", len(test_loader))

Test examples: 1532
Test batches: 6


In [5]:
# Inspect the shape of the first test batch
test_batch = next(iter(test_loader))

for key, value in test_batch.items():
    print(f"{key}: {tuple(value.shape)}")

# Share of unknown tokens in the test data (words not in the training vocabulary)
unk_id = vocab["<UNK>"]
text_fields = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

total_tokens = sum(len(example[field]) for example in test_encoded for field in text_fields)
unk_tokens = sum(
    token_id == unk_id
    for example in test_encoded
    for field in text_fields
    for token_id in example[field]
)

print(f"\nUnknown tokens in the test data: {unk_tokens} of {total_tokens} ({unk_tokens / total_tokens:.2%})")

observation_1: (256, 17)
observation_2: (256, 20)
hypothesis_1: (256, 28)
hypothesis_2: (256, 27)
label: (256,)

Unknown tokens in the test data: 888 of 61409 (1.45%)


## 4.4 Final Test Evaluation

Both loaded models are evaluated once on the held-out test set. No training is performed and no hyperparameters are changed.

The evaluation uses the existing `validate()` function from `src/training.py`, which was also used to compute the validation accuracy during training. It sets the model to `eval()` mode (dropout disabled), computes the `CrossEntropyLoss` and the accuracy, and runs without gradient computation. This makes the test results directly comparable with the validation results.

In [6]:
import torch.nn as nn
import pandas as pd

from src.training import validate

criterion = nn.CrossEntropyLoss()

test_results = []

for name, model, checkpoint in [
    ("Word Embeddings", we_model, we_checkpoint),
    ("RNN (GRU)", rnn_model, rnn_checkpoint),
]:
    model.eval()

    with torch.inference_mode():
        test_loss, test_acc = validate(model, test_loader, criterion, device)

    test_results.append({
        "Model": name,
        "Validation Accuracy": checkpoint["valid_accuracy"] * 100,
        "Test Accuracy": test_acc * 100,
        "Difference (Test - Validation)": (test_acc - checkpoint["valid_accuracy"]) * 100,
        "Test Loss": test_loss,
    })

test_results = pd.DataFrame(test_results)

test_results.style.format({
    "Validation Accuracy": "{:.2f} %",
    "Test Accuracy": "{:.2f} %",
    "Difference (Test - Validation)": "{:+.2f} pp",
    "Test Loss": "{:.4f}",
}).hide(axis="index")

Model,Validation Accuracy,Test Accuracy,Difference (Test - Validation),Test Loss
Word Embeddings,67.80 %,53.72 %,-14.08 pp,1.7117
RNN (GRU),66.10 %,53.13 %,-12.97 pp,2.4510


In [7]:
we_result, rnn_result = test_results.iloc[0], test_results.iloc[1]

for _, row in test_results.iterrows():
    print(
        f"{row['Model']:16s} | Test Accuracy: {row['Test Accuracy']:.2f} % "
        f"(Validation: {row['Validation Accuracy']:.2f} %, "
        f"{row['Difference (Test - Validation)']:+.2f} pp) | "
        f"Test Loss: {row['Test Loss']:.4f}"
    )

gap = we_result["Test Accuracy"] - rnn_result["Test Accuracy"]
better = "Word Embeddings" if gap > 0 else "RNN (GRU)"

print(f"\nHigher test accuracy: {better} ({abs(gap):.2f} percentage points difference)")
print(f"Number of test examples: {len(test_encoded)}")

Word Embeddings  | Test Accuracy: 53.72 % (Validation: 67.80 %, -14.08 pp) | Test Loss: 1.7117
RNN (GRU)        | Test Accuracy: 53.13 % (Validation: 66.10 %, -12.97 pp) | Test Loss: 2.4510

Higher test accuracy: Word Embeddings (0.59 percentage points difference)
Number of test examples: 1532
